# Building Entries — Total Visits, Most Visited Floor & Resources Per Person

**Question:** Given an `entries` table with `name`, `address`, `email`, `floor`, and `resources`, write a query (in both SQL and PySpark) that shows each person's **total visits**, **most visited floor** (the floor they visited most often), and a **comma-separated list of distinct resources** they used.

In [0]:
%sql
-- Step 1: Create the entries table tracking building visits:
--   name (person), address, email, floor visited, resource used (CPU/DESKTOP/MONITOR)
create table b_sql.b_practice.entries ( 
name varchar(20),
address varchar(20),
email varchar(20),
floor int,
resources varchar(10));

-- Step 2: Insert 6 sample entries for 2 people (A and B) across floors 1 and 2
insert into b_sql.b_practice.entries 
values ('A','Bangalore','A@gmail.com',1,'CPU'),('A','Bangalore','A1@gmail.com',1,'CPU'),('A','Bangalore','A2@gmail.com',2,'DESKTOP')
,('B','Bangalore','B@gmail.com',2,'DESKTOP'),('B','Bangalore','B1@gmail.com',2,'DESKTOP'),('B','Bangalore','B2@gmail.com',1,'MONITOR')

In [0]:
%sql
-- Verify the inserted data
select * from b_sql.b_practice.entries;

In [0]:
%sql
-- SQL Solution: Per-person total visits, most visited floor, and distinct resources used
--
-- CTE 1 (total_visit):    Group by name to get total visit count and comma-separated distinct resources
-- CTE 2 (floor_count):    Group by name+floor to get visit count per floor per person
-- CTE 3 (floor_rnk):     Rank floors per person by visit count (descending) so rnk=1 = most visited floor
-- Final query:            Join total_visit with floor_rnk (rnk=1) to get total visits, most visited floor, and resources
with total_visit as (select name,count(*)as total_count,listagg(distinct resources,',') WITHIN GROUP (ORDER BY resources) as resource_used from b_sql.b_practice.entries group by name),
floor_count as (select name,floor,count(*) as total_visit from b_sql.b_practice.entries group by name,floor),
floor_rnk as(select name,floor,total_visit,rank() over(partition by name order by total_visit desc) as rnk from floor_count)
select total_visit.name,total_count as total_visit,floor_rnk.floor as most_visited_floor,resource_used from total_visit join floor_rnk on total_visit.name=floor_rnk.name and floor_rnk.rnk=1
order by name,total_visit

In [0]:
# Import common PySpark SQL functions, types, and Window specification
# (Window is needed for partitionBy/orderBy to rank floors by visit count)
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window


In [0]:
# Load the entries table from Unity Catalog into a Spark DataFrame and display
df_entries=spark.read.table("b_sql.b_practice.entries")
df_entries.display()

In [0]:

# PySpark Solution: Same logic as the SQL CTE — total visits, most visited floor, distinct resources
#
# Part 1 — df_total: Group by name to get:
#   total_visit = count(*)
#   resources   = comma-separated distinct resources (sorted descending via collect_set + sort_array)
#
# Part 2 — df_flag: Group by name+floor to get visit count per floor,
#   then use row_number() window to rank floors (rnk=1 = most visited)
#
# Part 3 — Final: Join df_total with df_flag (rnk=1) to produce:
#   name | total_visit | most_visited_floor | resources

df_total = (
    df_entries
    .groupBy("name")
    .agg(
        count("*").alias("total_visit"),
        concat_ws(
            ",",
            sort_array(
                collect_set("resources"),
                asc=False
            )
        ).alias("resources")
    )
    .orderBy(col("total_visit").desc())
)

df_total.display()

# Rank floors per person by visit count
df_flag=df_entries.groupBy("name","floor").agg(count("*").alias("floor_visit"))
w=Window.partitionBy(col("name")).orderBy(col("floor_visit").desc())
df_flag=df_flag.withColumn("rnk",row_number().over(w)).filter(col("rnk")==1)
df_flag.display()

# Final join: total visits + most visited floor + resources
df_total.alias("t").join(df_flag.alias("f"),col("t.name")==col("f.name"),"inner").drop(col("f.name"),"rnk","floor_visit").display()
